In [6]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Any, TypedDict
import json
from pathlib import Path
from IPython.display import display
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Prefer the current working directory when launched from the project root.
# Set REPO_INTELLIGENCE_ROOT if the notebook is launched from another directory.
PROJECT_ROOT = Path(
    os.environ.get("REPO_INTELLIGENCE_ROOT", str(Path.cwd()))
).expanduser().resolve()
PROJECT_ROOT = Path("C:/self-study/ai/build/repository-intelligence/")
SRC_ROOT = PROJECT_ROOT / "src"

if not (SRC_ROOT / "repo_intelligence").is_dir():
    raise FileNotFoundError(
        "Could not find src/repo_intelligence under PROJECT_ROOT="
        f"{PROJECT_ROOT}. Set REPO_INTELLIGENCE_ROOT to the repository root."
    )

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from repo_intelligence.llm.config import load_env
from repo_intelligence.llm.openai_client import OpenAILLMClient
from repo_intelligence.planning import (
    InvestigationPlan,
    build_planning_request,
    load_architecture_context,
    plan_from_response,
    validate_plan,
)
from repo_intelligence.orchestration import (
    Orchestrator,
    RoleAssignment,
    assign_roles,
)
from repo_intelligence.multiagent import TaskPayload
from repo_intelligence.agent import JsonFileTraceSink
from repo_intelligence.repository_query import (
    M17_ORCHESTRATION_CONFIG,
    OFFLINE_COMMIT,
    QueryRouter,
    build_architect,
    build_engine,
    build_flow_investigator,
    build_memory_context,
    build_memory_manager,
    build_visualizer,
    default_artifacts_root,
    default_subject_repository,
    live_bindings,
    read_snapshot_metadata,
    resolve_read_target,
    verify_subject_snapshot,
)
from repo_intelligence.repository_query import (
    M17_ORCHESTRATION_CONFIG,
    OFFLINE_COMMIT,
    QueryRouter,
    build_architect,
    build_engine,
    build_flow_investigator,
    build_memory_context,
    build_memory_manager,
    build_visualizer,
    default_artifacts_root,
    default_subject_repository,
    live_bindings,
    read_snapshot_metadata,
    resolve_read_target,
    verify_subject_snapshot,
)
from repo_intelligence.planning import validate_plan
from repo_intelligence.llm.structured import validate_structured
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from repo_intelligence.llm.openai_client import OpenAILLMClient
from repo_intelligence.planning import PLANNING_SYSTEM_INSTRUCTION
load_env()
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\self-study\ai\build\repository-intelligence


In [7]:
from langchain_core.tools import tool

from repo_intelligence.tools.get_architecture_context import (
    get_architecture_context,
)
from repo_intelligence.tools.search_repository import (
    search_repository,
)
from repo_intelligence.tools.query_engineering_graph import (
    query_engineering_graph_tool,
)


@tool
def get_architecture_context_tool() -> str:
    """
    Retrieve the generated architecture context for the current repository snapshot.
    Returns architecture_context.md content and provenance metadata.
    """
    result = get_architecture_context({})

    if not result.success:
        return f"Error: {result.error.code} - {result.error.message}"

    return str(result.to_dict())


@tool
def search_repository_tool(
    query: str,
    path: str | None = None,
    max_results: int = 5,
) -> str:
    """
    Search the repository using semantic retrieval.

    Args:
        query: Natural language search query.
        path: Optional repository-relative path filter.
        max_results: Maximum results to return.
    """
    arguments = {
        "query": query,
        "max_results": max_results,
    }

    if path:
        arguments["path"] = path

    result = search_repository(arguments)

    if not result.success:
        return f"Error: {result.error.code} - {result.error.message}"

    return str(result.to_dict())


@tool
def query_engineering_graph_tool_wrapper(
    start_node_id: str | None = None,
    direction: str = "out",
    relationship_kinds: list[str] | None = None,
    node_kinds: list[str] | None = None,
    max_depth: int = 1,
) -> str:
    """
    Query the engineering graph.

    Examples:
    - Call without start_node_id to get node-kind summary.
    - Call with start_node_id to traverse graph relationships.
    """
    arguments = {}

    if start_node_id:
        arguments["start_node_id"] = start_node_id

    if direction:
        arguments["direction"] = direction

    if relationship_kinds:
        arguments["relationship_kinds"] = relationship_kinds

    if node_kinds:
        arguments["node_kinds"] = node_kinds

    arguments["max_depth"] = max_depth

    result = query_engineering_graph_tool(arguments)

    if not result.success:
        return f"Error: {result.error.code} - {result.error.message}"

    return str(result.to_dict())


tools = [
    get_architecture_context_tool,
    search_repository_tool,
    query_engineering_graph_tool_wrapper,
]

In [8]:
repository_root, readable_target, resolve_error = resolve_read_target()

if resolve_error:
    raise RuntimeError(f"Could not resolve subject repository: {resolve_error}")

verified_commit, commit_source, verification_error = verify_subject_snapshot(
    repository_root
)
if verification_error:
    raise RuntimeError(
        "Subject snapshot verification failed; refusing to analyze it: "
        f"{verification_error}"
    )

snapshot_metadata = read_snapshot_metadata(repository_root)
repository_label = (
    snapshot_metadata.get("name")
    or repository_root.name
    or str(repository_root)
)
current_commit = verified_commit or OFFLINE_COMMIT

print(f"Repository root: {repository_root}")
print(f"Repository scope: {repository_label}")
print(f"Commit scope: {current_commit}")
print(f"Commit source: {commit_source}")
print(f"Readable entry target: {readable_target}")

Repository root: C:\self-study\ai\build\repository-intelligence\artifacts\main_2026-09-22_ef11dac\repository\sample-Agentic-Ai-Data-Operations
Repository scope: sample-Agentic-Ai-Data-Operations
Commit scope: ef11dac8618b6344d6a5e4add6ef51cd2811d7e0
Commit source: git
Readable entry target: README.md


In [9]:
architecture_context, architecture_context_record = load_architecture_context(
    output_root=default_artifacts_root()
)
print("Architecture Context:", architecture_context_record)
print("Shared runtime initialized.")

Architecture Context: {'artifact': 'artifacts\\main_2026-09-22_ef11dac\\architecture_context\\architecture_context.md', 'schema_version': 'architecture_context.v1', 'commit_sha': 'ef11dac8618b6344d6a5e4add6ef51cd2811d7e0', 'snapshot_id': 'main_2026-09-22_ef11dac'}
Shared runtime initialized.


In [10]:
class AgentAssignments(BaseModel):
    """Readable preview: plan step ID -> concrete specialist slot."""

    agent_assignments: dict[str, str] = Field(default_factory=dict)


class RepoIntelligenceState(TypedDict, total=False):
    objective: str
    plan: InvestigationPlan
    role_assignment: list[RoleAssignment]
    agent_assignments: AgentAssignments
    orchestration_run: Any
    execution_status: str
    execution_error: str

In [11]:
planner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
#planner_llm = OpenAILLMClient(model="gpt-4o-mini")

plan_agent = create_agent(
    model=planner_llm,
    tools=tools,
    system_prompt= PLANNING_SYSTEM_INSTRUCTION,
    response_format=InvestigationPlan,
)

In [12]:
result = plan_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "How do the agents, tools, and MCP servers interact?"
            ),
        }
    ]
})

investigation_plan: InvestigationPlan = result["structured_response"]

print(investigation_plan)
print(investigation_plan.objective)
print(investigation_plan.status)
if investigation_plan is None:
    raise StructuredOutputValidationError(
        "planner response contained no JSON object"
    )
for step in investigation_plan.steps:
   print(step.id, step.goal, step.status, step.result)

objective='Understand the interaction between agents, tools, and MCP servers.' steps=[PlanStep(id='P1', goal='Retrieve the architecture context to gather initial information about agents, tools, and MCP servers.', status='pending', result=None), PlanStep(id='P2', goal='Investigate the specific roles and functionalities of agents within the architecture.', status='pending', result=None), PlanStep(id='P3', goal='Examine the tools used in the architecture and their interactions with agents and MCP servers.', status='pending', result=None), PlanStep(id='P4', goal='Analyze the communication protocols and data flow between agents, tools, and MCP servers.', status='pending', result=None)] status='pending'
Understand the interaction between agents, tools, and MCP servers.
pending
P1 Retrieve the architecture context to gather initial information about agents, tools, and MCP servers. pending None
P2 Investigate the specific roles and functionalities of agents within the architecture. pending No

In [ ]:
investigator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
from repo_intelligence.agents.architecture_investigator import build_architecture_investigator_prompt,ARCHITECTURE_INVESTIGATOR_PROMPT_V1
from repo_intelligence.agents.models.architecture_report import ArchitectureReport

prompt = build_architecture_investigator_prompt(
        question="Retrieve the architecture context to understand the current setup of agents, tools, and MCP servers",
        repository_scope=repository_label,
        commit_scope=current_commit,
        instructions = ARCHITECTURE_INVESTIGATOR_PROMPT_V1
    )
investigator_agent = create_agent(
    model=investigator_llm,
    tools=tools,
    system_prompt= prompt,
    response_format=ArchitectureReport,
)

investigation_report = investigator_agent.invoke({"messages": "Execute investigation plan."})
print(investigation_report)

c:\self-study\ai\build\repository-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2450.37it/s]


OpenAIContextOverflowError: Error code: 400 - {'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 128215 tokens (including 947 in the response_format schemas.). Please reduce the length of the messages or schemas.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}